## Natural Language Processing Spring 2026
---
# Assembling the MiniMind Transformer

**Objective:** Build a modern Large Language Model (LLM) from scratch using pure PyTorch. We will construct the Rotary Position Embeddings (RoPE), Grouped Query Attention (GQA), the SwiGLU FeedForward Network, and finally assemble the Transformer blocks into a Causal Language Model.

*Note: This notebook is designed to be run sequentially. Each implementation step includes theory, mathematical formulas, and unit tests to verify the tensor shapes and math operations before moving on to the next component.*

## Step 0: Setup and Recap
First, let's import PyTorch and reload our model's hyperparameter configuration and the `RMSNorm` (Root Mean Square Normalization) module, both of which were introduced and implemented during our last session.

As a quick reminder, `RMSNorm` is preferred over standard `LayerNorm` because it removes the mean-centering step, making it computationally cheaper while maintaining training stability.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

Let's load our `MiniMindConfig` and the `RMSNorm` implementation from the previous session to serve as the foundation for our new components.

In [ ]:
# 1. Model Configuration
class MiniMindConfig:
    def __init__(self):
        self.vocab_size = 6400
        self.hidden_size = 768
        self.num_hidden_layers = 2  # Kept small for rapid testing
        self.num_attention_heads = 8
        self.num_key_value_heads = 4  # GQA: 4 KV heads for 8 Query heads
        self.head_dim = self.hidden_size // self.num_attention_heads
        self.intermediate_size = 2048
        self.max_position_embeddings = 2048
        self.rms_norm_eps = 1e-6
        self.dropout = 0.0
        self.tie_word_embeddings = True

config = MiniMindConfig()
print(f"Config loaded. Hidden size: {config.hidden_size}, Q_Heads: {config.num_attention_heads}, KV_Heads: {config.num_key_value_heads}")

In [ ]:
# 2. RMSNorm Implementation
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        # Learnable scaling parameter
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # torch.rsqrt calculates 1 / sqrt(x)
        normed = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return (self.weight * normed.float()).type_as(x)

In [ ]:
# Set global testing variables
batch_size, seq_len = 2, 10
dummy_x = torch.randn(batch_size, seq_len, config.hidden_size)
print(f"Dummy input shape: {dummy_x.shape}")

## Step 1: Review and Warm-up — The Evolution of Position Embeddings

### Theory & Concepts
Before we jump into building the Attention mechanism, let's review why the industry abandoned the classic Absolute Positional Embeddings used in the original 2017 'Attention is All You Need' paper.

**1. Standard Absolute Positional Embedding (The Old Way)**
$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

* **How it works:** These sine/cosine vectors are simply **added** to the input token embeddings at the very beginning of the network: $X_{input} = \text{Word}_{Embeddings} + PE$.
* **The Flaw:** As the sequence passes through multiple deep layers, this absolute positional information gets diluted. Furthermore, attention cares about the *relative distance* between two words (e.g., "how far is 'apple' from 'eating'?"), but adding absolute positions makes relative calculations mathematically messy.


**2. Rotary Position Embedding (RoPE - The Modern Way)**
The standard formulation applies a rotation matrix to the query/key vectors. For a vector $x$ at position $m$, we multiply pairs of its dimensions by a rotation matrix:

$$ f_q(x_m, m) = \begin{pmatrix} \cos(m\theta_1) & -\sin(m\theta_1) & 0 & 0 & \cdots \\ \sin(m\theta_1) & \cos(m\theta_1) & 0 & 0 & \cdots \\ 0 & 0 & \cos(m\theta_2) & -\sin(m\theta_2) & \cdots \\ 0 & 0 & \sin(m\theta_2) & \cos(m\theta_2) & \cdots \\ \vdots & \vdots & \vdots & \vdots & \ddots \end{pmatrix} \begin{pmatrix} x_m^{(1)} \\ x_m^{(2)} \\ x_m^{(3)} \\ x_m^{(4)} \\ \vdots \end{pmatrix} $$

* **How it works:** RoPE **multiplies (rotates)** the Query and Key vectors *inside* the attention layer. It pairs up the features of the vector and rotates them in a 2D plane by an angle $m\theta_i$, where $m$ is the token's position and $\theta_i$ is the frequency for that dimension pair.
* **The Benefit:** Thanks to trigonometric identities, the dot product of a Query at position $m$ and a Key at position $n$ will depend strictly on their relative distance $m-n$. It's mathematically elegant and extrapolates better to longer sequences.



### PyTorch Functions Used:
* `torch.outer(a, b)`: Computes the outer product of two vectors.
* `torch.cat(...)`: Concatenates tensors along a given dimension.
* `unsqueeze(...)`: Adds a dimension of size 1 (used here for broadcasting across batch and head dimensions).

In [ ]:
def precompute_freqs_cis(dim: int, end: int, rope_base: float = 1e6):
    # 1. Inverse frequencies (theta)
    freqs = 1.0 / (rope_base ** (torch.arange(0, dim, 2).float() / dim))
    # 2. Position sequence 'm'
    t = torch.arange(end, device=freqs.device)
    # 3. Outer product: m * theta
    freqs = torch.outer(t, freqs).float()
    # 4. Duplicate the frequencies to match head_dim
    freqs_cos = torch.cat([torch.cos(freqs), torch.cos(freqs)], dim=-1)
    freqs_sin = torch.cat([torch.sin(freqs), torch.sin(freqs)], dim=-1)
    return freqs_cos, freqs_sin

def apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1):
    # Helper to apply the [-x2, x1] transformation (mimicking complex multiplication)
    def rotate_half(x):
        half_dim = x.shape[-1] // 2
        return torch.cat((-x[..., half_dim:], x[..., :half_dim]), dim=-1)

    # Add dimensions so [seq_len, head_dim] broadcasts to [batch, seq_len, heads, head_dim]
    cos_unsqueezed = cos.unsqueeze(0).unsqueeze(2)
    sin_unsqueezed = sin.unsqueeze(0).unsqueeze(2)

    q_embed = (q * cos_unsqueezed) + (rotate_half(q) * sin_unsqueezed)
    k_embed = (k * cos_unsqueezed) + (rotate_half(k) * sin_unsqueezed)
    return q_embed, k_embed


In [ ]:
# --- UNIT TEST 1 ---
print("--- Testing RoPE ---")
dummy_q = torch.randn(batch_size, seq_len, config.num_attention_heads, config.head_dim)
dummy_k = torch.randn(batch_size, seq_len, config.num_key_value_heads, config.head_dim)

pos_embs = precompute_freqs_cis(config.head_dim, seq_len)
q_rot, k_rot = apply_rotary_pos_emb(dummy_q, dummy_k, pos_embs[0], pos_embs[1])

print(f"Original Q shape: {dummy_q.shape} | Rotated Q shape: {q_rot.shape}")
print(f"Original K shape: {dummy_k.shape} | Rotated K shape: {k_rot.shape}")
assert q_rot.shape == dummy_q.shape, "RoPE changed Q shape!"
assert k_rot.shape == dummy_k.shape, "RoPE changed K shape!"
print("RoPE implementation successfully rotated tensors without changing shapes.")

### How RoPE Scales to N-Dimensions (A 6D Example)

Let's trace how the `rotate_half` trick elegantly scales to any even-dimensional vector. Suppose we have a 6-dimensional feature vector:

$$X = [x_1, x_2, x_3, x_4, x_5, x_6]$$

In this 6D space, we need 3 different rotation frequencies (angles), which we will call $\theta_1, \theta_2, \theta_3$.

#### Step 1: Prepare and Concatenate the Angles
The code calculates the 3 base frequencies and concatenates them to match the 6D shape:
* **Angle Vector:** $\theta = [\theta_1, \theta_2, \theta_3]$
* **Cos Vector:** $Cos = [\cos\theta_1, \cos\theta_2, \cos\theta_3, \cos\theta_1, \cos\theta_2, \cos\theta_3]$
* **Sin Vector:** $Sin = [\sin\theta_1, \sin\theta_2, \sin\theta_3, \sin\theta_1, \sin\theta_2, \sin\theta_3]$

*Notice how $\theta_1$ aligns with the 1st and 4th items, $\theta_2$ aligns with the 2nd and 5th items, and $\theta_3$ aligns with the 3rd and 6th items. This is how the code "forces" the pairing.*

#### Step 2: The `rotate_half` Function
We split the vector $X$ in half, negate the second half, and swap their positions:

$$rotate\_half(X) = [-x_4, -x_5, -x_6, x_1, x_2, x_3]$$

#### Step 3: Vector Multiplication and Addition
Now we execute the core line of code: `q_embed = (X * Cos) + (rotate_half(X) * Sin)`

**1. Left Term (`X * Cos`):**
$$[x_1\cos\theta_1, \quad x_2\cos\theta_2, \quad x_3\cos\theta_3, \quad x_4\cos\theta_1, \quad x_5\cos\theta_2, \quad x_6\cos\theta_3]$$

**2. Right Term (`rotate_half(X) * Sin`):**
$$[-x_4\sin\theta_1, \quad -x_5\sin\theta_2, \quad -x_6\sin\theta_3, \quad x_1\sin\theta_1, \quad x_2\sin\theta_2, \quad x_3\sin\theta_3]$$

**3. Add them together:**
1. $x_1\cos\theta_1 - x_4\sin\theta_1$
2. $x_2\cos\theta_2 - x_5\sin\theta_2$
3. $x_3\cos\theta_3 - x_6\sin\theta_3$
4. $x_4\cos\theta_1 + x_1\sin\theta_1$
5. $x_5\cos\theta_2 + x_2\sin\theta_2$
6. $x_6\cos\theta_3 + x_3\sin\theta_3$

#### Step 4: The Mathematical Magic
If we look at the results by extracting their engineered "pairs", we get perfect 2D rotation matrices for every pair simultaneously:

* **Pair 1 (Items 1 & 4, using $\theta_1$):**
  * New $x_1 = x_1\cos\theta_1 - x_4\sin\theta_1$
  * New $x_4 = x_4\cos\theta_1 + x_1\sin\theta_1$
* **Pair 2 (Items 2 & 5, using $\theta_2$):**
  * New $x_2 = x_2\cos\theta_2 - x_5\sin\theta_2$
  * New $x_5 = x_5\cos\theta_2 + x_2\sin\theta_2$
* **Pair 3 (Items 3 & 6, using $\theta_3$):**
  * New $x_3 = x_3\cos\theta_3 - x_6\sin\theta_3$
  * New $x_6 = x_6\cos\theta_3 + x_3\sin\theta_3$

**Conclusion:** By simply concatenating the frequencies and shifting the tensor halves, PyTorch can perform 64 (or any arbitrary number) independent 2D rotations simultaneously in a single, highly optimized hardware step. No `for` loops or slow memory slicing are required!

## Step 2: Grouped Query Attention (GQA)



### Theory & Concepts
In Standard Multi-Head Attention (MHA), if we have 8 Query heads, we have 8 Key heads and 8 Value heads. Storing 8 Keys and Values for every generated token takes up massive GPU memory (this is called the KV Cache).

**Grouped Query Attention (GQA)** solves this memory bottleneck. We keep a large number of Query heads (e.g., 8), but group them to share a smaller number of Key/Value heads (e.g., 4). This means exactly 2 Query heads share 1 Key head and 1 Value head, cutting KV Cache memory usage in half with almost zero loss in performance.

**The Attention Math:**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_{head}}}\right)V$$

Because Q has 8 heads and K/V have 4, we must mathematically "repeat" the K/V heads to perform dot-product attention. We will create a `repeat_kv` helper function to do this efficiently.

### PyTorch Functions Used:
* `tensor.expand(...)`: Creates a new view of the tensor with repeated elements without actually allocating new memory.
* `tensor.reshape(...)`: Changes the shape of the tensor. Combined with `expand`, we interleave the repeated heads.
* `tensor.transpose(dim0, dim1)`: Swaps two dimensions. We use this to bring the `head` dimension forward for batch matrix multiplication.
* `tensor.triu(diagonal)`: Returns the upper triangular part of a matrix. We use this to create the causal mask (setting future tokens to `-inf` so the model cannot look ahead).

In [ ]:
def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    bs, slen, num_kv_heads, head_dim = x.shape
    if n_rep == 1: return x
    # Expand adds the n_rep dimension: [B, S, 4, 2, 96]
    x_expanded = x[:, :, :, None, :].expand(bs, slen, num_kv_heads, n_rep, head_dim)
    # Reshape collapses the heads and reps together: [B, S, 8, 96]
    return x_expanded.reshape(bs, slen, num_kv_heads * n_rep, head_dim)

### `repeat_kv` Example

1. Assume in the MiniMind model, the input tensor `x` (K or V) has an initial shape of `[2, 10, 4, 96]`:

`batch_size (bs) = 2`
`seq_len (slen) = 10`
`num_kv_heads = 4` (Original number of K and V heads)
`head_dim = 96`
`n_rep = 2` (Since Q_heads(8)÷KV_heads(4)=2, each KV head needs to be shared by 2 Q heads)

2. Step-by-Step Transformation

**Step 1**: Adding a Dummy Dimension
`x[:, :, :, None, :]`
Inserts a new dimension of size 1 after num_kv_heads (equivalent to unsqueeze).

Shape Change: `[2, 10, 4, 96] → [2, 10, 4, 1, 96]`

**Step 2**: Memory-Efficient Expansion (Expand)
.expand(bs, slen, num_kv_heads, n_rep, head_dim)
Broadcasts the newly inserted dimension to the size of n_rep.

Shape Change: `[2, 10, 4, 1, 96] → [2, 10, 4, 2, 96]`

Engineering Trick: PyTorch's expand operation only creates a "view" and does not actually copy data in memory. This keeps the underlying VRAM usage unchanged, saving massive amounts of memory bandwidth.

**Step 3**: Reshaping and Aligning (Reshape)
`.reshape(bs, slen, num_kv_heads * n_rep, head_dim)`
Merges the "original heads" dimension with the "repetition" dimension to create the final heads dimension.

Merge Logic: `4×2=8`

Shape Change: `[2, 10, 4, 2, 96] → [2, 10, 8, 96]`

**Final Result**

After processing, the shape of the K or V tensor becomes `[2, 10, 8, 96]`, perfectly matching the number of heads in the Query tensor. They are now ready for efficient dot-product attention calculations (`Q @ K^T`).

Now let's implement the Attention module.

In [ ]:
class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_local_heads = config.num_attention_heads
        self.n_local_kv_heads = config.num_key_value_heads
        self.n_rep = self.n_local_heads // self.n_local_kv_heads
        self.head_dim = config.head_dim

        # Linear layers with NO bias (modern standard, why?)
        self.q_proj = nn.Linear(config.hidden_size, self.n_local_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.hidden_size, self.n_local_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.hidden_size, self.n_local_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.n_local_heads * self.head_dim, config.hidden_size, bias=False)

        self.q_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.k_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)

    def forward(self, x, position_embeddings):
        bsz, seq_len, _ = x.shape

        # 1. Projections
        xq, xk, xv = self.q_proj(x), self.k_proj(x), self.v_proj(x)

        # 2. Reshape to separate heads
        xq = xq.view(bsz, seq_len, self.n_local_heads, self.head_dim)
        xk = xk.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)
        xv = xv.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)

        # 3. Norm Q and K
        xq, xk = self.q_norm(xq), self.k_norm(xk)

        # 4. Apply RoPE to Queries and Keys (not Values!)
        cos, sin = position_embeddings
        xq, xk = apply_rotary_pos_emb(xq, xk, cos, sin)

        # 5. Repeat KV and Transpose for Math [B, Heads, Seq, Head_Dim]
        xq = xq.transpose(1, 2)
        xk = repeat_kv(xk, self.n_rep).transpose(1, 2)
        xv = repeat_kv(xv, self.n_rep).transpose(1, 2)

        # 6. Attention Math: Softmax((Q @ K^T) / sqrt(d)) @ V
        scores = (xq @ xk.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # Causal Mask (hide future tokens)
        mask = torch.full((seq_len, seq_len), float("-inf"), device=scores.device).triu(1)
        scores = scores + mask

        attention_output = F.softmax(scores, dim=-1) @ xv

        # 7. Final reshaping: contiguous() ensures memory layout is correct before view/reshape
        attention_output = attention_output.transpose(1, 2).contiguous().reshape(bsz, seq_len, -1)
        return self.o_proj(attention_output)

### Structure and Calculation Flow of the Attention Class

The Attention class is the core engine of the Transformer. In MiniMind, it implements Grouped Query Attention (GQA) combined with Rotary Position Embedding (RoPE).

Let's break down exactly what happens during the forward pass, step-by-step, tracking the tensor shapes.

1. Assumptions for this example:

`batch_size (bsz) = 2`
`seq_len = 10`
`hidden_size = 768`
`n_local_heads (Q) = 8`
`n_local_kv_heads (K/V) = 4`
`head_dim = 96 (Since 768÷8=96)`
`n_rep = 2 (Since 8÷4=2)`
`Input Shape: [2, 10, 768]`

2. Linear Projections (`q_proj`, `k_proj`, `v_proj`)

The input is projected into Queries (Q), Keys (K), and Values (V) using independent linear layers (without biases).

`xq = self.q_proj(x) → Shape: [2, 10, 768] (8 heads × 96)`

`xk = self.k_proj(x) → Shape: [2, 10, 384] (4 heads × 96)`

`xv = self.v_proj(x) → Shape: [2, 10, 384] (4 heads × 96)`

3. Reshaping for Multi-Head Operations

To perform operations on individual heads, we separate the hidden_size dimension into num_heads and head_dim.

`xq.view(bsz, seq_len, self.n_local_heads, self.head_dim) → Shape: [2, 10, 8, 96]`

`xk.view(...) → Shape: [2, 10, 4, 96]`

`xv.view(...) → Shape: [2, 10, 4, 96]`

Note: After reshaping, Q and K are normalized using RMSNorm.

4. Applying Rotary Position Embeddings (RoPE)

We inject positional information by rotating the features of Q and K in a 2D plane. Crucially, RoPE is NOT applied to Values (V).

`xq, xk = apply_rotary_pos_emb(xq, xk, cos, sin)`

Shapes remain unchanged: Q is `[2, 10, 8, 96]`, K is `[2, 10, 4, 96]`.

5. Transpose and GQA KV-Repetition

For PyTorch's batch matrix multiplication (@), the dimensions that interact must be the last two (seq_len and head_dim). Therefore, we transpose the seq_len and head dimensions.
Additionally, because Q has 8 heads and K/V have 4, we must "stretch" (repeat) K and V to match Q before we can multiply them.

`xq = xq.transpose(1, 2) → Shape: [2, 8, 10, 96]`

`xk = repeat_kv(xk, self.n_rep).transpose(1, 2) → Shape: [2, 8, 10, 96]`

`xv = repeat_kv(xv, self.n_rep).transpose(1, 2) → Shape: [2, 8, 10, 96]'
5. The Attention Math

Now we execute the core formula: $$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_{head}}}\right)V$$

A. Compute Scores:

`scores = (xq @ xk.transpose(-2, -1)) / math.sqrt(self.head_dim)`

Shape computation: `[2, 8, 10, 96] @ [2, 8, 96, 10] = [2, 8, 10, 10]`

This `10×10` matrix represents how much every token attends to every other token.

B. Causal Masking:

We add a mask containing negative infinity (-inf) to the upper triangle of the `10×10` matrix. This prevents tokens from looking "forward" into the future during next-token prediction.

C. Softmax & Value Multiplication:

`attention_output = F.softmax(scores, dim=-1) @ xv`

The softmax converts the scores into probabilities (summing to 1).

Shape computation: `[2, 8, 10, 10] @ [2, 8, 10, 96] = [2, 8, 10, 96]`

6. Final Reshaping and Output Projection

Finally, we merge the heads back together and project the result to the original hidden size.

`attention_output = attention_output.transpose(1, 2).contiguous().reshape(bsz, seq_len, -1)`

Shape changes: `[2, 8, 10, 96] → [2, 10, 8, 96] → [2, 10, 768]`

`output = self.o_proj(attention_output) → Final Shape: [2, 10, 768]`

In [ ]:
# --- UNIT TEST 2 ---
print("--- Testing GQA Attention ---")
attn_layer = Attention(config)
attn_out = attn_layer(dummy_x, pos_embs)
print(f"Attention output shape: {attn_out.shape}")
assert attn_out.shape == dummy_x.shape, f"Expected {dummy_x.shape}, got {attn_out.shape}"
print("GQA implementation successful.")

## Step 3: SwiGLU FeedForward Network


### Theory & Concepts
The original Transformer used a simple two-layer MLP: $$\text{Linear}(\text{ReLU}(\text{Linear}(x)))$$

Modern LLMs (like LLaMA, Qwen, and MiniMind) replace this with a gated architecture called **SwiGLU**. We create two separate high-dimensional pathways from the input: one acts as an information 'gate' via a SiLU (Swish) activation, which is then multiplied element-wise against the other un-activated pathway.

**Math Equation:**
$$\text{SwiGLU}(x) = \text{Down}_{Proj}\Big( \text{SiLU}(\text{Gate}_{Proj}(x)) \odot \text{Up}_{Proj}(x) \Big)$$

Notice that the input takes *two* paths (Gate and Up), they are multiplied together element-wise ($\odot$), and then projected back down. This acts as a dynamic information filter.

### PyTorch Functions Used:
* `nn.Linear(in, out, bias=False)`: Standard fully connected layer without bias (modern LLMs drop biases to improve stability and training speed).
* `F.silu(x)`: The Sigmoid Linear Unit activation function ($x * \sigma(x)$).

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        # Project from 768 to 2048
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        # Project back from 2048 to 768
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)

    def forward(self, x):
        # Element-wise multiplication of the activated gate and the up projection
        gated_features = F.silu(self.gate_proj(x)) * self.up_proj(x)
        return self.down_proj(gated_features)

### FFNN Example

1. Assumptions for this example:

`batch_size (bsz) = 2`
`seq_len = 10`
`hidden_size = 768`
`intermediate_size = 2048` (This is the expanded hidden dimension inside the FFN)
Input Shape: `[2, 10, 768]`

2. Dual Linear Projections (`gate_proj` and `up_proj`)

Unlike a traditional MLP that projects the input once, SwiGLU splits the input into two parallel pathways. Both projections expand the dimension from hidden_size to intermediate_size without using biases.

`gate_out = self.gate_proj(x) → Shape: [2, 10, 2048]`
`up_out = self.up_proj(x) → Shape: [2, 10, 2048]`

3. The Activation Function (SiLU / Swish)

We apply the SiLU (Sigmoid Linear Unit) activation function only to the gate pathway. SiLU is defined as $x⋅\sigma(x)$, which provides a smooth, non-monotonic curve that helps gradients flow better in deep networks.

`activated_gate = F.silu(gate_out) → Shape remains: [2, 10, 2048]`

4. Element-wise Multiplication (The "Gating" Mechanism)

This is where the magic happens. We multiply the activated_gate and the up_out together element-by-element ($\odot$).

`gated_features = activated_gate * up_out → Shape remains: [2, 10, 2048]`

Intuition: The activated_gate acts as a dynamic filter. For every single feature in the 2048-dimensional space, the gate decides what percentage of the up_out information is allowed to pass through to the next stage.

5. Linear Down Projection (`down_proj`)

Finally, we take these filtered, high-dimensional features and project them back down to the original hidden dimension so they can be added back to the residual stream.

`output = self.down_proj(gated_features)`

Shape computation: `[2, 10, 2048] @ [2048, 768] → Final Shape: [2, 10, 768]`

In [ ]:
# --- UNIT TEST 3 ---
print("--- Testing SwiGLU FFN ---")
ffn_layer = FeedForward(config)
ffn_out = ffn_layer(attn_out)
print(f"FFN output shape: {ffn_out.shape}")
assert ffn_out.shape == attn_out.shape, "FFN changed tensor shape!"
print("SwiGLU implementation successful.")

---
## Step 4: Assembling the Pre-Norm Transformer Block

### Theory & Concepts
How do we stack these layers? Originally, transformers normalized *after* the residual addition (Post-Norm). Modern LLMs use **Pre-Norm**: we normalize the input *before* passing it into the sub-layers, and add the *un-normalized* residual.

```text
x_out = x_in + Attention(Norm(x_in))
x_final = x_out + FFN(Norm(x_out))
```

**Why?** This ensures there is a clear, unmodified mathematical 'highway' (the `+` operations) from the first layer all the way to the end of the network. This prevents vanishing gradients in very deep networks, making training much more stable.

In [ ]:
class MiniMindBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.self_attn = Attention(config)
        self.input_layernorm = RMSNorm(config.hidden_size)
        self.post_attention_layernorm = RMSNorm(config.hidden_size)
        self.mlp = FeedForward(config)

    def forward(self, x, position_embeddings):
        # --- First Sub-Layer: Attention ---
        residual = x
        normed_x = self.input_layernorm(x)
        attn_out = self.self_attn(normed_x, position_embeddings)
        x = residual + attn_out # Add residual

        # --- Second Sub-Layer: FFN ---
        residual = x
        normed_x = self.post_attention_layernorm(x)
        ffn_out = self.mlp(normed_x)
        x = residual + ffn_out # Add residual

        return x

In [ ]:
# --- UNIT TEST 4 ---
print("--- Testing Transformer Block ---")
block = MiniMindBlock(config)
block_out = block(dummy_x, pos_embs)
print(f"Block forward pass successful. Output shape: {block_out.shape}")
assert block_out.shape == dummy_x.shape, "Block changed tensor shape!"
print("Transformer Block assembly successful.")

## Step 5: Final Assembly — Model & LM Head


### Theory & Concepts
We are at the finish line! We need to:
1. Generate initial token embeddings.
2. Pass them sequentially through our stacked blocks.
3. Build the Language Modeling Head (LM Head) to predict the next word in the vocabulary.

**Weight Tying Trick:**
We will set the `lm_head` weights to point to the exact same memory as the `embed_tokens` weights. If a specific vector represents the word 'Apple' going in, that same vector pattern should score highest for 'Apple' going out. This saves us `vocab_size * hidden_size` parameters!

**Calculating Loss (Next-Token Prediction):**
To train the model, we shift the logits to the left (`[:-1]`) and the labels to the right (`[1:]`).
*Why?* Because the logit at position $t$ is trying to predict the ground truth label at position $t+1$.

### PyTorch Functions Used:
* `nn.ModuleList([...])`: Holds submodules in a list. Standard Python lists do not register parameters in PyTorch.
* `tensor.contiguous()`: Ensures the tensor is stored in a contiguous chunk of memory before reshaping.
* `F.cross_entropy(..., ignore_index=-100)`: Calculates the loss. We use `ignore_index=-100` so that padded tokens or user-prompts (during Supervised Fine-Tuning) are not penalized.

In [ ]:
class MiniMindForCausalLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        # 1. The embedding layer
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)

        # 2. Stack the blocks
        self.layers = nn.ModuleList([MiniMindBlock(config) for _ in range(config.num_hidden_layers)])

        # 3. Final Norm
        self.norm = RMSNorm(config.hidden_size)

        # 4. LM Head: Maps hidden states back to vocabulary probabilities
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)

        # WEIGHT TYING!
        if config.tie_word_embeddings:
            self.embed_tokens.weight = self.lm_head.weight

    def forward(self, input_ids, position_embeddings, labels=None):
        # Embed tokens -> [B, S, hidden_size]
        x = self.embed_tokens(input_ids)

        # Pass through layers
        for layer in self.layers:
            x = layer(x, position_embeddings)

        # Final normalization
        x = self.norm(x)

        # Map back to vocabulary -> [B, S, vocab_size]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            # SHIFT LOGITS AND LABELS for next-token prediction
            # If input is [A, B, C], we want to predict [B, C, D].
            # So the logit at pos 0 (for A) is matched with label at pos 1 (B).

            # Drop the last logit prediction
            shift_logits = logits[..., :-1, :].contiguous()
            # Drop the first label
            shift_labels = labels[..., 1:].contiguous()

            # Calculate Cross Entropy Loss (flattening batch and seq_len dims)
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100 # Ignore padding tokens
            )

        return logits, loss


In [ ]:
# --- UNIT TEST 5 ---
print("--- Testing Full Causal LM ---")
lm = MiniMindForCausalLM(config)
dummy_input_ids = torch.randint(0, config.vocab_size, (batch_size, seq_len))
dummy_labels = torch.randint(0, config.vocab_size, (batch_size, seq_len))

logits, loss = lm(dummy_input_ids, pos_embs, labels=dummy_labels)
print(f"Logits shape: {logits.shape} (Expected: [{batch_size}, {seq_len}, {config.vocab_size}])")
print(f"Loss computed successfully: {loss.item():.4f}")
print("Architecture Assembly Complete!")

## Step 6: Wrap-up & Next Steps


**Review the Architecture Flow:**
`Token IDs` $\rightarrow$ `Embedding` $\rightarrow$ `[RMSNorm -> GQA(RoPE) -> Add] * N` $\rightarrow$ `[RMSNorm -> SwiGLU -> Add] * N` $\rightarrow$ `RMSNorm` $\rightarrow$ `LM Head` $\rightarrow$ `CrossEntropyLoss`.